In [8]:
import pandas as pd
import psycopg2

sheet_curr = '1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk'
sheet_archive = '1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE'
gid_matches = '2141931777'
gid_deck = '590005429'

# MATCH_ID      = 11000000000
# EVENT_ID      = 12000000000
# DECK_ID       = 13000000000
# EVENT_TYPE_ID = 14000000000
# LOAD_RPT_ID   = 15000000000
# EV_REJ_ID     = 16000000000
# MATCH_REJ_ID  = 17000000000

In [9]:
with open("credentials.txt", "r") as file:
    credentials = [line.strip() for line in file]

In [10]:
def conn(query, credentials=credentials):
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        cursor.execute(query)

        conn.commit()
    except psycopg2.Error as e:
        print('Error:', e)
    finally:
        if conn:
            cursor.close()
            conn.close()

In [11]:
def delete_table(TABLE):
    query = f'DROP TABLE IF EXISTS "{TABLE}" CASCADE'
    conn(query)

In [12]:
def create_new_tables():
    create_valid_decks_query = """
    CREATE TABLE IF NOT EXISTS "VALID_DECKS" (
        "FORMAT" VARCHAR(30),
        "ARCHETYPE" VARCHAR(30),
        "SUBARCHETYPE" VARCHAR(30),
        "DECK_ID" BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 13000000000) PRIMARY KEY,
        "PROC_DT" TIMESTAMP WITHOUT TIME ZONE,
        CONSTRAINT unique_deck UNIQUE ("FORMAT", "ARCHETYPE", "SUBARCHETYPE")
    );
    """
    create_valid_event_types_query = """
    CREATE TABLE IF NOT EXISTS "VALID_EVENT_TYPES" (
        "FORMAT" VARCHAR(30),
        "EVENT_TYPE" VARCHAR(30),
        "EVENT_TYPE_ID" BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 14000000000) PRIMARY KEY,
        "PROC_DT" TIMESTAMP WITHOUT TIME ZONE,
        CONSTRAINT unique_event_type UNIQUE ("FORMAT", "EVENT_TYPE")
    );
    """
    create_events_query = """
    CREATE TABLE IF NOT EXISTS "EVENTS" (
        "EVENT_ID" BIGINT PRIMARY KEY,
        "EVENT_DATE" DATE,
        "EVENT_TYPE_ID" BIGINT,
        "PROC_DT" TIMESTAMP WITHOUT TIME ZONE,
        FOREIGN KEY ("EVENT_TYPE_ID") REFERENCES "VALID_EVENT_TYPES"("EVENT_TYPE_ID") ON UPDATE CASCADE
    );
    """
    create_matches_query = """
    CREATE TABLE IF NOT EXISTS "MATCHES" (
        "MATCH_ID" BIGINT,
        "P1" VARCHAR(30),
        "P2" VARCHAR(30),
        "P1_WINS" INT,
        "P2_WINS" INT,
        "MATCH_WINNER" VARCHAR(2),
        "P1_DECK_ID" BIGINT,
        "P2_DECK_ID" BIGINT,
        "P1_NOTE" VARCHAR(100),
        "P2_NOTE" VARCHAR(100),
        "EVENT_ID" BIGINT,
        "PROC_DT" TIMESTAMP WITHOUT TIME ZONE,
        PRIMARY KEY ("MATCH_ID", "P1"),
        FOREIGN KEY ("P1_DECK_ID") REFERENCES "VALID_DECKS"("DECK_ID") ON UPDATE CASCADE,
        FOREIGN KEY ("P2_DECK_ID") REFERENCES "VALID_DECKS"("DECK_ID") ON UPDATE CASCADE,
        FOREIGN KEY ("EVENT_ID") REFERENCES "EVENTS"("EVENT_ID") ON UPDATE CASCADE ON DELETE CASCADE
    );
    """
    create_load_reports_query = """
    CREATE TABLE IF NOT EXISTS "LOAD_REPORTS" (
        "LOAD_RPT_ID" BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 15000000000) PRIMARY KEY,
        "START_DATE" DATE,
        "END_DATE" DATE,
        "RECORDS_FULL_DS" INT,
        "RECORDS_TOTAL" INT,
        "EVENTS_IGNORED" INT,
        "RECORDS_PROC" INT,
        "MATCHES_DELETED" INT,
        "MATCHES_PROC" INT,
        "MATCHES_SKIPPED" INT,
        "EVENTS_DELETED" INT,
        "EVENTS_PROC" INT,
        "EVENTS_SKIPPED" INT,
        "DB_CONN_ERROR_TEXT" VARCHAR(30),
        "PROC_DT" TIMESTAMP WITHOUT TIME ZONE
    );
    """
    create_event_rejections_query = """
    CREATE TABLE IF NOT EXISTS "EVENT_REJECTIONS" (
        "EVENT_REJ_ID" BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 16000000000) PRIMARY KEY,
        "LOAD_RPT_ID" BIGINT,
        "EVENT_ID" BIGINT,
        "EVENT_DATE" DATE,
        "EVENT_TYPE_ID" BIGINT,
        "PROC_DT" TIMESTAMP WITHOUT TIME ZONE,
        "REJ_TYPE" VARCHAR(1),
        "EVENT_REJ_TEXT" VARCHAR(30),
        FOREIGN KEY ("LOAD_RPT_ID") REFERENCES "LOAD_REPORTS"("LOAD_RPT_ID") ON UPDATE CASCADE ON DELETE CASCADE
    );
    """
    create_match_rejections_query = """
    CREATE TABLE IF NOT EXISTS "MATCH_REJECTIONS" (
        "MATCH_REJ_ID" BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 17000000000) PRIMARY KEY,
        "LOAD_RPT_ID" BIGINT,
        "MATCH_ID" BIGINT,
        "P1" VARCHAR(30),
        "P2" VARCHAR(30),
        "P1_WINS" INT,
        "P2_WINS" INT,
        "MATCH_WINNER" VARCHAR(2),
        "P1_DECK_ID" BIGINT,
        "P2_DECK_ID" BIGINT,
        "P1_NOTE" VARCHAR(30),
        "P2_NOTE" VARCHAR(30),
        "EVENT_ID" BIGINT,
        "PROC_DT" TIMESTAMP WITHOUT TIME ZONE,
        "REJ_TYPE" VARCHAR(1),
        "MATCH_REJ_TEXT" VARCHAR(30),
        FOREIGN KEY ("LOAD_RPT_ID") REFERENCES "LOAD_REPORTS"("LOAD_RPT_ID") ON UPDATE CASCADE ON DELETE CASCADE
    );
    """
    create_fkey_indexes = """
    CREATE INDEX idx_events_event_type_id ON "EVENTS"("EVENT_TYPE_ID");
    CREATE INDEX idx_matches_p1_deck_id ON "MATCHES"("P1_DECK_ID");
    CREATE INDEX idx_matches_p2_deck_id ON "MATCHES"("P2_DECK_ID");
    CREATE INDEX idx_matches_event_id ON "MATCHES"("EVENT_ID");
    CREATE INDEX idx_event_rejections_load_rpt_id ON "EVENT_REJECTIONS"("LOAD_RPT_ID");
    CREATE INDEX idx_match_rejections_load_rpt_id ON "MATCH_REJECTIONS"("LOAD_RPT_ID");
    """
    try:
        conn(create_valid_decks_query)
        conn(create_valid_event_types_query)
        conn(create_events_query)
        conn(create_matches_query)
        conn(create_load_reports_query)
        conn(create_event_rejections_query)
        conn(create_match_rejections_query)
        conn(create_fkey_indexes)
    except:
        print('Exception. No tables created.')

In [13]:
delete_table('VALID_DECKS')
delete_table('VALID_EVENT_TYPES')
delete_table('EVENTS')
delete_table('MATCHES')
delete_table('LOAD_REPORTS')
delete_table('EVENT_REJECTIONS')
delete_table('MATCH_REJECTIONS')

In [14]:
create_new_tables()